# 03. Text Vectorization

This notebook converts the cleaned news corpus into numerical representations for downstream clustering.

## Purpose
The aim of this notebook is to:
- load the final cleaned dataset
- generate a **TF-IDF** representation of the corpus
- generate **S-BERT embeddings** for semantic representation
- save the resulting feature outputs for later stages of the pipeline

## Methods used
- **TF-IDF** for sparse lexical features
- **S-BERT (`all-MiniLM-L6-v2`)** for dense semantic embeddings

## Expected input
This notebook expects the cleaned dataset produced during preprocessing:

```python
combined_clean_deduplicated.xlsx
```

## Expected outputs
The notebook produces:
- `combined_tfidf_matrix.pkl`
- `combined_tfidf_features.pkl`
- `sbert_embeddings.npy`

**Note:** Generated feature files are not included in the public repository due to size constraints.


## 1. Import libraries

This section imports the required libraries for vectorisation and file export.


In [ ]:
import pandas as pd
import numpy as np
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

## 2. Define input file



In [ ]:
input_file = "combined_clean_deduplicated.xlsx" 

## 3. Load the cleaned dataset

In [ ]:
df = pd.read_excel(input_file)

print("Dataset shape:", df.shape)
print("Available columns:", df.columns.tolist())

## 4. Prepare text data

This project uses the `clean_text` column generated during preprocessing.


In [ ]:
texts = df["clean_text"].astype(str).tolist()

print("Number of documents:", len(texts))

## 5. TF-IDF vectorisation

TF-IDF captures lexical importance by weighting terms according to their frequency within a document and across the corpus.

For this project, the vectoriser uses:
- `max_features=10000`
- English stopword removal


In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
tfidf_features = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of TF-IDF features:", len(tfidf_features))

## 6. Save TF-IDF outputs

The sparse matrix and feature vocabulary are saved separately for reuse in later notebooks.


In [ ]:
with open("combined_tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

with open("combined_tfidf_features.pkl", "wb") as f:
    pickle.dump(tfidf_features, f)

print("Saved: combined_tfidf_matrix.pkl")
print("Saved: combined_tfidf_features.pkl")

## 7. S-BERT embeddings

S-BERT is used to generate dense semantic embeddings for each article using the pre-trained `all-MiniLM-L6-v2` model.

If `sentence-transformers` is not installed in your environment, install it once before running the notebook:

```python
!pip install -U sentence-transformers
```


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

sbert_embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True
)

print("S-BERT embedding shape:", sbert_embeddings.shape)

## 8. Save S-BERT output

In [ ]:
np.save("sbert_embeddings.npy", sbert_embeddings)

print("Saved: sbert_embeddings.npy")

## 9. Summary

At this point:
- **TF-IDF** provides a sparse lexical representation of the corpus
- **S-BERT** provides a dense semantic representation of the corpus

These outputs are used in the next stage for dimensionality reduction and clustering experiments.
